# Testing

Evaluate the pretrained model against the 700 labelled images and exercise the API. Run `python main.py evaluate` first so `artifacts/predictions.csv` exists — this notebook reads that rather than re-running inference.

In [ ]:
import sys
from pathlib import Path

BACKEND_ROOT = Path.cwd().parents[2]
sys.path[:0] = [str(BACKEND_ROOT), str(BACKEND_ROOT / "src")]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

from prayaas.config.configuration import CLASS_LABELS, settings

predictions = pd.read_csv(settings.artifacts_dir / "predictions.csv")
predictions.head()

## The OPMD problem

The model emits 3 classes but only 2 folders exist on disk. OPMD has no ground truth, so scoring requires collapsing it into one of the two — and that choice moves the numbers a lot.

In [ ]:
crosstab = pd.crosstab(
    predictions["folder"], predictions["predicted_raw"].map(dict(enumerate(CLASS_LABELS)))
)
crosstab

OPMD predictions land almost entirely on CANCER images, which says OPMD cases were filed under CANCER when this dataset was labelled. Hence `OPMD_IS_CANCER=true`.

In [ ]:
for opmd_is_cancer in (True, False):
    collapsed = predictions["predicted_raw"].map(
        lambda c: (0 if opmd_is_cancer else 1) if c == 2 else c
    )
    accuracy = (collapsed == predictions["label"]).mean()
    recall = (
        collapsed[predictions["label"] == 0] == 0
    ).mean()
    print(f"OPMD_IS_CANCER={str(opmd_is_cancer):5s} accuracy={accuracy:.4f}  CANCER recall={recall:.4f}")

## Classification report (current setting)

In [ ]:
y_true, y_pred = predictions["label"], predictions["predicted"]

print(f"OPMD_IS_CANCER = {settings.opmd_is_cancer}\n")
print(classification_report(y_true, y_pred, target_names=["CANCER", "NON CANCER"], zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=["CANCER", "NON CANCER"], cmap="Blues"
)
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

## False negatives

CANCER images called NON CANCER. In a screening tool these are the costly errors — worth reviewing individually.

In [ ]:
missed = predictions[(predictions["label"] == 0) & (predictions["predicted"] == 1)]
print(f"{len(missed)} of {(predictions['label'] == 0).sum()} CANCER images missed")
missed[["path", "confidence"]].head(10)

In [ ]:
from PIL import Image

if len(missed):
    fig, axes = plt.subplots(1, 4, figsize=(14, 4))
    for ax, (_, row) in zip(axes, missed.head(4).iterrows()):
        with Image.open(row["path"]) as im:
            ax.imshow(im)
        ax.set_title(f"missed ({row['confidence']:.0%})", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Confidence distribution — is the model confidently wrong?

In [ ]:
predictions["correct"] = predictions["label"] == predictions["predicted"]

sns.histplot(data=predictions, x="confidence", hue="correct", bins=25, element="step")
plt.title("Confidence, correct vs incorrect")
plt.tight_layout()
plt.show()

predictions.groupby("correct")["confidence"].describe()[["count", "mean", "50%"]]

## API smoke test (in-process, no server needed)

In [ ]:
from fastapi.testclient import TestClient

from app import app

client = TestClient(app)
print(client.get("/health").json())

In [ ]:
image_path = Path(predictions["path"].iloc[0])

with image_path.open("rb") as f:
    response = client.post("/predict", files={"file": (image_path.name, f, "image/jpeg")})

print(response.status_code)
response.json()

In [ ]:
# Legacy base64 contract, as used by the original Streamlit client.
import base64

encoded = base64.b64encode(image_path.read_bytes()).decode()
print(client.post("/predict/base64", json={"file": encoded}).json())